# Child Health Trend Forecasting
### Using ARIMA (Auto-Regressive Integrated Moving Average)

**Input Features (Monthly Aggregates — Age 0–6):**
| Column | Description |
|---|---|
| `Avg_BMI` | Average BMI |
| `Avg_Weight_kg` | Average weight (kg) |
| `Avg_Height_cm` | Average height (cm) |
| `Pct_Dewormed` | % of children dewormed |
| `Pct_Vitamins_Intake` | % receiving vitamins |
| `Pct_Immunization` | % immunized |
| `Pct_Vaccination` | % vaccinated |

**Output (Forecast Target):**
| Column | Description |
|---|---|
| `Pct_Improved_Next_Month` | % of children predicted to improve |

> Forecast mode: **Next N Months** — forecast all indicators up to 60 months ahead (default: 12)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Step 1: Load the Dataset

In [ ]:
df = pd.read_csv('child_health_trend.csv')
print('Shape:', df.shape)
print('Date range:', df['Month_Label'].iloc[0], 'to', df['Month_Label'].iloc[-1])
df.head(10)

In [ ]:
df.describe()

## Step 2: Visualize Historical Time Series

In [ ]:
numeric_cols = [
    'Avg_BMI', 'Avg_Weight_kg', 'Avg_Height_cm',
    'Pct_Dewormed', 'Pct_Vitamins_Intake',
    'Pct_Immunization', 'Pct_Vaccination',
    'Pct_Improved_Next_Month'
]

fig, axes = plt.subplots(4, 2, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].plot(df['Month_No'], df[col], color='steelblue', linewidth=1.8)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Month No.')
    axes[i].grid(True, alpha=0.3)
plt.suptitle('Child Health Trends — Jan 2020 to Dec 2025', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step 3: Stationarity Check (ADF Test)
ARIMA needs stationary data. If p-value > 0.05, differencing (d≥1) is applied automatically.

In [ ]:
print(f"{'Column':<35} | {'p-value':>8} | Status")
print('-' * 70)
for col in numeric_cols:
    p = adfuller(df[col].dropna())[1]
    status = 'Stationary' if p <= 0.05 else 'Non-Stationary (d=1 applied by ARIMA)'
    print(f"{col:<35} | {p:>8.4f} | {status}")

## Step 4: Train ARIMA Models (Auto Order Selection)

In [ ]:
targets = [
    'Avg_BMI', 'Avg_Weight_kg', 'Avg_Height_cm',
    'Pct_Dewormed', 'Pct_Vitamins_Intake',
    'Pct_Immunization', 'Pct_Vaccination',
    'Pct_Improved_Next_Month'
]

arima_models = {}
best_orders  = {}

print('Training ARIMA models...\n')
for col in targets:
    model = auto_arima(
        df[col].values,
        seasonal=False,
        stepwise=True,
        information_criterion='aic',
        suppress_warnings=True,
        error_action='ignore'
    )
    arima_models[col] = model
    best_orders[col]  = model.order
    print(f"  {col:<38} ARIMA{model.order}   AIC: {model.aic():.2f}")

print('\nAll models trained!')

## Step 5: Model Evaluation (In-Sample MAE / RMSE / MAPE)

In [ ]:
print(f"{'Column':<38} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>8}")
print('-' * 72)
for col in targets:
    actual = df[col].values
    fitted = arima_models[col].predict_in_sample()
    n      = min(len(actual), len(fitted))
    mae    = mean_absolute_error(actual[:n], fitted[:n])
    rmse   = np.sqrt(mean_squared_error(actual[:n], fitted[:n]))
    mape   = np.mean(np.abs((actual[:n] - fitted[:n]) / actual[:n])) * 100
    print(f"  {col:<36} | {mae:>8.4f} | {rmse:>8.4f} | {mape:>7.2f}%")

## Step 6: Forecast — Next N Months
Change `FORECAST_MONTHS` to any value from **1 to 60**. Default is **12**.

In [ ]:
# ── SET HOW MANY MONTHS TO FORECAST (1–60) ────────────────
FORECAST_MONTHS = 12
# ──────────────────────────────────────────────────────────

MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

def next_labels(last_label, n):
    m, y   = last_label.split('-')
    mi, yr = MONTH_NAMES.index(m), int(y)
    labels = []
    for _ in range(n):
        mi += 1
        if mi >= 12:
            mi, yr = 0, yr + 1
        labels.append(f"{MONTH_NAMES[mi]}-{yr}")
    return labels

last_label    = df['Month_Label'].iloc[-1]
last_no       = int(df['Month_No'].max())
future_labels = next_labels(last_label, FORECAST_MONTHS)

forecast_df = pd.DataFrame({
    'Month_No':    range(last_no + 1, last_no + FORECAST_MONTHS + 1),
    'Month_Label': future_labels
})

for col in targets:
    fc = arima_models[col].predict(n_periods=FORECAST_MONTHS)
    forecast_df[col] = np.round(fc, 2)

print(f'Forecast for the next {FORECAST_MONTHS} months:\n')
forecast_df

## Step 7: Plot Historical + N-Month Forecast (All Indicators)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(targets):
    ax   = axes[i]
    std  = df[col].std() * 0.3
    fc_x = forecast_df['Month_No']
    fc_y = forecast_df[col]

    ax.plot(df['Month_No'], df[col], color='steelblue', label='Historical', linewidth=1.8)
    ax.plot(fc_x, fc_y, color='tomato', linestyle='--', label=f'Forecast ({FORECAST_MONTHS}mo)', linewidth=1.8)
    ax.fill_between(fc_x, fc_y - std, fc_y + std, color='tomato', alpha=0.15, label='Confidence')
    ax.axvline(x=last_no, color='gray', linestyle=':', alpha=0.6)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('Month No.')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'ARIMA Forecast — Next {FORECAST_MONTHS} Months (Age 0–6)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step 8: Focus Plot — Pct_Improved_Next_Month

In [ ]:
col  = 'Pct_Improved_Next_Month'
fc_x = forecast_df['Month_No']
fc_y = forecast_df[col]
std  = df[col].std() * 0.3

plt.figure(figsize=(14, 5))
plt.plot(df['Month_No'], df[col], color='steelblue', linewidth=2, label='Historical')
plt.plot(fc_x, fc_y, color='tomato', linestyle='--', linewidth=2, label=f'Forecast ({FORECAST_MONTHS} months)')
plt.fill_between(fc_x, fc_y - std, fc_y + std, color='tomato', alpha=0.15, label='Confidence Band')
plt.axvline(x=last_no, color='gray', linestyle=':', alpha=0.7, label='Forecast Start')

plt.annotate(f"+1mo: {fc_y.iloc[0]:.1f}%",
             xy=(fc_x.iloc[0], fc_y.iloc[0]),
             xytext=(fc_x.iloc[0] + 0.5, fc_y.iloc[0] + 1.2),
             fontsize=9, color='tomato',
             arrowprops=dict(arrowstyle='->', color='tomato', lw=1))
plt.annotate(f"+{FORECAST_MONTHS}mo: {fc_y.iloc[-1]:.1f}%",
             xy=(fc_x.iloc[-1], fc_y.iloc[-1]),
             xytext=(fc_x.iloc[-1] - 4, fc_y.iloc[-1] + 1.2),
             fontsize=9, color='tomato',
             arrowprops=dict(arrowstyle='->', color='tomato', lw=1))

plt.title('% Children Predicted to Improve — Historical + Forecast', fontsize=13, fontweight='bold')
plt.xlabel('Month No.')
plt.ylabel('% Improved')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Last historical  ({last_label})  : {df[col].iloc[-1]:.2f}%")
print(f"Month +1         ({future_labels[0]}) : {fc_y.iloc[0]:.2f}%")
print(f"Month +{FORECAST_MONTHS}        ({future_labels[-1]}): {fc_y.iloc[-1]:.2f}%")

## Step 9: Save All ARIMA Models as Pickle

In [ ]:
bundle = {
    'models':           arima_models,
    'best_orders':      best_orders,
    'targets':          targets,
    'last_month_no':    last_no,
    'last_month_label': last_label,
    'history':          df
}

with open('arima_models.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print('arima_models.pkl saved!')
print('Stored models:', list(arima_models.keys()))

## Step 10: Test Loading Pickle

In [ ]:
with open('arima_models.pkl', 'rb') as f:
    loaded = pickle.load(f)

col     = 'Pct_Improved_Next_Month'
fc      = loaded['models'][col].predict(n_periods=FORECAST_MONTHS)
labels  = next_labels(loaded['last_month_label'], FORECAST_MONTHS)

print(f'Forecast for {col} — next {FORECAST_MONTHS} months:')
for lbl, val in zip(labels, fc):
    print(f'  {lbl}: {val:.2f}%')